In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import torch
from torchtext._torchtext import (
    Vocab as VocabPybind,
)

from scgpt.tokenizer.gene_tokenizer import GeneVocab
from scgpt.model import TransformerGenerator, TransformerModel
from scgpt.utils import load_pretrained, set_seed, map_raw_id_to_vocab_id, compute_perturbation_metrics

import sys
sys.path.append("../")
from model import scGPTForPerturbationResponsePrediction

from gears import PertData, GEARS
from gears.inference import compute_metrics, deeper_analysis, non_dropout_analysis
from gears.utils import create_cell_graph_dataset_for_prediction

In [ ]:
def set_seed(seed):
    """set random seed."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

data_name = "adamson"
split = "simulation"
batch_size = 4
eval_batch_size = 4

print("Loading data...")
pert_data = PertData("./data")
pert_data.load(data_name=data_name)

In [ ]:
# settings for the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.backends.mps.is_available():
    device = torch.device("mps")

embsize = 512  # embedding dimension
d_hid = 512  # dimension of the feedforward network model in nn.TransformerEncoder
nlayers = 12  # number of nn.TransformerEncoderLayer in nn.TransformerEncoder
nhead = 8  # number of heads in nn.MultiheadAttention
n_layers_cls = 3
dropout = 0  # dropout probability
use_fast_transformer = True  # whether to use fast transformer

# settings for vocab
pad_token = "<pad>"
special_tokens = [pad_token, "<cls>", "<eoc>"]
pad_value = 0  # for padding values
pert_pad_id = 0
include_zero_gene = "all"
max_seq_len = 1536

load_model = "../pretrained_weights/scGPT_human"
if load_model is not None:
    model_dir = Path(load_model)
    model_config_file = model_dir / "args.json"
    model_file = model_dir / "best_model.pt"
    vocab_file = model_dir / "vocab.json"

    vocab = GeneVocab.from_file(vocab_file)
    for s in special_tokens:
        if s not in vocab:
            vocab.append_token(s)

    pert_data.adata.var["id_in_vocab"] = [
        1 if gene in vocab else -1 for gene in pert_data.adata.var["gene_name"]
    ]
    gene_ids_in_vocab = np.array(pert_data.adata.var["id_in_vocab"])
    print(
        f"match {np.sum(gene_ids_in_vocab >= 0)}/{len(gene_ids_in_vocab)} genes "
        f"in vocabulary of size {len(vocab)}."
    )
    genes = pert_data.adata.var["gene_name"].tolist()

    # model
    with open(model_config_file, "r") as f:
        model_configs = json.load(f)
    print(
        f"Resume model from {model_file}, the model args will override the "
        f"config {model_config_file}."
    )
    embsize = model_configs["embsize"]
    nhead = model_configs["nheads"]
    d_hid = model_configs["d_hid"]
    nlayers = model_configs["nlayers"]
    n_layers_cls = model_configs["n_layers_cls"]
else:
    genes = pert_data.adata.var["gene_name"].tolist()
    vocab = Vocab(
        VocabPybind(genes + special_tokens, None)
    )  # bidirectional lookup [gene <-> int]
vocab.set_default_index(vocab["<pad>"])
gene_ids = np.array(
    [vocab[gene] if gene in vocab else vocab["<pad>"] for gene in genes], dtype=int
)
n_genes = len(genes)

ntokens = len(vocab)  # size of vocabulary

model = TransformerGenerator(
    ntokens,
    embsize,
    nhead,
    d_hid,
    nlayers,
    nlayers_cls=n_layers_cls,
    n_cls=1,
    vocab=vocab,
    dropout=dropout,
    pad_token=pad_token,
    pad_value=pad_value,
    pert_pad_id=pert_pad_id,
    use_fast_transformer=use_fast_transformer,
)
load_pretrained(model, torch.load(model_file, map_location=device), verbose=False)
model.eval()

embedding_model = TransformerModel(
        ntoken=len(vocab),
        d_model=model_configs["embsize"],
        nhead=model_configs["nheads"],
        d_hid=model_configs["d_hid"],
        nlayers=model_configs["nlayers"],
        nlayers_cls=model_configs["n_layers_cls"],
        n_cls=1,
        vocab=vocab,
        dropout=model_configs["dropout"],
        pad_token=model_configs["pad_token"],
        pad_value=model_configs["pad_value"],
        do_mvc=True,
        do_dab=False,
        use_batch_labels=False,
        domain_spec_batchnorm=False,
        explicit_zero_prob=False,
        use_fast_transformer=use_fast_transformer,
        fast_transformer_backend="flash",
        pre_norm=False,
    )

load_pretrained(embedding_model, torch.load(model_file, map_location=device), verbose=False)
embedding_model.eval()

In [ ]:
model

In [ ]:
nano_model = scGPTForPerturbationResponsePrediction.from_pretrained("scGPT_human")
nano_model.eval()

In [ ]:
print(torch.equal(nano_model.scgpt.pert_encoder.weight, model.pert_encoder.weight))

nano_model.scgpt.pert_encoder.weight.data.copy_(model.pert_encoder.weight.data)

print(torch.equal(nano_model.scgpt.pert_encoder.weight, model.pert_encoder.weight))

# print(f"model pert_encoder weight: {model.pert_encoder.weight}")

# print(f"nano_model pert_encoder weight: {nano_model.scgpt.pert_encoder.weight}")

In [ ]:
print(all(torch.equal(p1, p2) for p1, p2 in zip(model.decoder.bias_decoder.parameters(), nano_model.decoder.bias_decoder.parameters())))
print(all(torch.equal(p1, p2) for p1, p2 in zip(model.decoder.coeff_decoder.parameters(), nano_model.decoder.coeff_decoder.parameters())))

nano_model.decoder.bias_decoder.load_state_dict({k.replace('fc.', ''): v for k, v in model.decoder.bias_decoder.state_dict().items()})
nano_model.decoder.coeff_decoder.load_state_dict({k.replace('fc.', ''): v for k, v in model.decoder.coeff_decoder.state_dict().items()})


print(all(torch.equal(p1, p2) for p1, p2 in zip(model.decoder.bias_decoder.parameters(), nano_model.decoder.bias_decoder.parameters())))
print(all(torch.equal(p1, p2) for p1, p2 in zip(model.decoder.coeff_decoder.parameters(), nano_model.decoder.coeff_decoder.parameters())))

In [ ]:
test_gene_ids = torch.tensor([[1, 2, 3], [2, 3, 4], [19, 20, 382]], dtype=torch.long)
test_gene_values = torch.tensor([[0.1, 0.2, 0.3], [0.2, 0.3, 0.4], [0.3, 0.4, 0.5]], dtype=torch.float)
test_pert_flags = torch.tensor([[0, 1, 0], [0, 0, 0], [0, 0, 1]], dtype=torch.long)
test_src_key_padding_mask = torch.zeros_like(test_gene_values, dtype=torch.bool)

model_encodings = model._encode(test_gene_ids, test_gene_values, input_pert_flags=test_pert_flags, src_key_padding_mask=test_src_key_padding_mask)
nano_encodings = nano_model.scgpt(test_gene_ids, test_gene_values, src_key_padding_mask=test_src_key_padding_mask, pert_labels=test_pert_flags)

same_encodings = torch.allclose(model_encodings, nano_encodings, atol=1e-5)

model_output = model(test_gene_ids, test_gene_values, test_pert_flags, src_key_padding_mask=test_src_key_padding_mask)["mlm_output"]
nano_model_output = nano_model(test_gene_ids, test_gene_values, src_key_padding_mask=test_src_key_padding_mask, pert_labels=test_pert_flags)

same_output = torch.allclose(model_output, nano_model_output, atol=1e-5)

print(f"Encodings are the same: {same_encodings}")
print(f"Outputs are the same: {same_output}")

In [ ]:
from perturbation_data import PerturbationDataset, PerturbationDataSplitter
from scGPT_tokenizer import scGPTTokenizer

from torch.utils.data import DataLoader


adata = pert_data.adata
adata.var['gene_symbol'] = adata.var['gene_name']

tokenizer = scGPTTokenizer.from_pretrained("scGPT_human")
tokenizer.max_length = 1536
data_splitter = PerturbationDataSplitter(adata, tokenizer, seed=42)
train_adata, val_adata, test_adata = data_splitter.get_train_val_test()

train_dataset = PerturbationDataset(train_adata, tokenizer, split='train')
test_dataset = PerturbationDataset(test_adata, tokenizer, split='test')
val_dataset = PerturbationDataset(val_adata, tokenizer, split='val')
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=train_dataset.collate_fn)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, collate_fn=test_dataset.collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=val_dataset.collate_fn)

In [ ]:
for batch in train_loader:
    print(batch.keys())
    break

In [ ]:
test_dataset.perturbations

In [ ]:
test_set_perts = []
for p in test_dataset.perturbations:
    if p != "ctrl":
        ps = p.split("+")
        if ps[0] != 'ctrl':
            test_set_perts.append(ps[0])
        elif ps[1] != 'ctrl':
            test_set_perts.append(ps[1])

print("Test set perturbations:", test_set_perts)
pert_data.prepare_split(split=split, seed=1, only_test_set_perts=True, test_pert_genes=test_set_perts)
pert_data.get_dataloader(batch_size=batch_size, test_batch_size=eval_batch_size)

In [ ]:
from pyparsing import Dict


def eval_perturb(
    loader: DataLoader, model: TransformerGenerator, device: torch.device
) -> Dict:
    """
    Run model in inference mode using a given data loader
    """

    model.eval()
    model.to(device)
    pert_cat = []
    pred = []
    truth = []
    pred_de = []
    truth_de = []
    results = {}
    logvar = []

    for itr, batch in enumerate(loader):
        batch.to(device)
        pert_cat.extend(batch.pert)

        with torch.no_grad():
            p = model.pred_perturb(
                batch,
                include_zero_gene=include_zero_gene,
                gene_ids=gene_ids,
            )
            t = batch.y
            pred.extend(p.cpu())
            truth.extend(t.cpu())

            # Differentially expressed genes
            for itr, de_idx in enumerate(batch.de_idx):
                pred_de.append(p[itr, de_idx])
                truth_de.append(t[itr, de_idx])
            
    results["pert_cat"] = np.array(pert_cat)
    pred = torch.stack(pred) # (N, n_genes)
    truth = torch.stack(truth) # (N, n_genes)
    results["pred"] = pred.detach().cpu().numpy()
    results["truth"] = truth.detach().cpu().numpy()

    pred_de = torch.stack(pred_de) # (N, n_de_genes)
    truth_de = torch.stack(truth_de)
    results["pred_de"] = pred_de.detach().cpu().numpy()
    results["truth_de"] = truth_de.detach().cpu().numpy()

    return results

og_result = eval_perturb(
    pert_data.dataloader["test_loader"], model, device
)

In [ ]:
# Check data loading.
for batch_data, nano_data in zip(pert_data.dataloader['test_loader'], val_loader):

    batch_size = len(batch_data.pert)
    x = batch_data.x
    ori_gene_values = x[:, 0].view(batch_size, -1)  # (batch_size, n_genes)
    pert_flags = x[:, 1].long().view(batch_size, -1)
    input_gene_ids = torch.arange(ori_gene_values.size(1))
    input_values = ori_gene_values[:, input_gene_ids]
    input_pert_flags = pert_flags[:, input_gene_ids]
    mapped_input_gene_ids = map_raw_id_to_vocab_id(input_gene_ids, gene_ids)
    mapped_input_gene_ids = mapped_input_gene_ids.repeat(batch_size, 1)
    perts = batch_data.pert

    nano_gene_ids = nano_data['gene_ids']  # (batch_size, n_genes)
    nano_gene_values = nano_data['gene_values']  # (batch_size, n_gen
    nano_pert_flags = nano_data['pert_labels']  # (batch_size, n_genes)
    nano_target_gene_values = nano_data['target_values']  # (batch_size, n_genes)
    target_gene_values = batch_data.y  # (batch_size, n_genes)
    nano_perts = nano_data['perturbations']  # (batch_size, n_genes)

    # Check if the gene_ids match the original gene names in adata.var['gene_name']
    print("Checking gene_ids match...")
    print(torch.equal(nano_gene_ids, torch.tensor(mapped_input_gene_ids, dtype=torch.long)))
    print("og: ", mapped_input_gene_ids)
    print("nano: ", nano_gene_ids)

    print("Checking perturbation ...")
    print("og: ", perts)
    print("nano: ", nano_perts)

    print("Checking gene_values match...")
    print(torch.allclose(nano_gene_values, input_values, atol=1e-5))
    print("og: ", input_values)
    print("nano: ", nano_gene_values)

    print("Checking pert_flags match...")
    print(torch.equal(nano_pert_flags, input_pert_flags))

    break